# B1 query/hierarchy improvement
Run the single cell with T4 selected. Original notebook stopped at Drive mount, before training. Complete Drive/dataset authentication using the account holding Phase3_B1_Input.zip.
Eight windows of up to 384 tokens; four learned decoder queries; encoder attention/mean and decoder feature fusion; small ordered document BiGRU; partial updates to the last two encoder/decoder layers. No score gain has been measured.
Four-hour training wall-time budget, additional preparation/export, automatic 75-minute deadline reserve. Only rolling latest.pt and selected best.pt are written in a new run. Other runs and inputs are untouched.


In [ ]:
# SINGLE B1 IMPROVEMENT CELL. Run this cell instead of the old training cells.
# Your own Phase1 B1 input is reused; new training goes to its own Drive folder.
import os,sys,time,json,hashlib,shutil,subprocess,importlib.util,base64,io,zipfile,gc
from pathlib import Path
for proc in Path('/proc').iterdir():
 if not proc.name.isdigit():continue
 try:args=proc.joinpath('cmdline').read_bytes().split(b'\0')
 except (FileNotFoundError,PermissionError,ProcessLookupError):continue
 if b'phase4_cjpe.runtime' in args and b'--worker' in args:raise RuntimeError('A Phase4 worker is already running here, PID '+proc.name+'. Do not start a second worker.')
for module,package in [('ijson','ijson>=3.2,<4'),('huggingface_hub','huggingface_hub>=0.26,<2'),('sklearn','scikit-learn>=1.4,<2'),('tokenizers','tokenizers')]:
 if importlib.util.find_spec(module) is None:subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
import torch
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > T4 GPU.'
print('ACTUAL GPU:',torch.cuda.get_device_name(0),flush=True)
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
 try:drive.mount('/content/drive',force_remount=True)
 except Exception as exc:raise RuntimeError('Drive mount failed. Complete the Google permission prompt with the account holding your B1 input. The model has not started training. Original error: '+str(exc)) from exc
assert Path('/content/drive/MyDrive').is_dir(),'Drive is not mounted; no training started.'
PERSIST=Path('/content/drive/MyDrive/CS781_Phase4')
WORK=Path('/content/phase4_work');WORK.mkdir(parents=True,exist_ok=True)
assert shutil.disk_usage('/content').free>12e9,'Need at least 12 GB free temporary Colab disk.'
RUN_ROOT=PERSIST/'runs/02_B1_Partial_QueryHier';RUN_ROOT.mkdir(parents=True,exist_ok=True)
INPUT_NAME='Phase3_B1_Input.zip'
INPUT=next((p for p in (PERSIST/'inputs'/INPUT_NAME,PERSIST/INPUT_NAME,Path('/content/drive/MyDrive')/INPUT_NAME,Path('/content/inputs')/INPUT_NAME,Path('/content')/INPUT_NAME) if p.is_file()),None)
assert INPUT is not None,'Upload the existing Phase3_B1_Input.zip to this account: MyDrive/CS781_Phase4/inputs/. Importing a notebook does not transfer its model weights.'
CODE_ROOT=Path('/content/phase4_b1_query_hierarchy_code')
source_bytes=base64.b64decode('...')
assert hashlib.sha256(source_bytes).hexdigest()=='360bfdaa23588b0893e597fe6f1056a078c8587af0cbe3d3361bcd59bf2c8897'
with zipfile.ZipFile(io.BytesIO(source_bytes)) as z:
 assert z.testzip() is None
 for name in z.namelist():
  p=Path(name);assert not p.is_absolute() and '..' not in p.parts
  dest=CODE_ROOT/p;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(z.read(name))
sys.path[:]=[str(CODE_ROOT)]+[p for p in sys.path if p!=str(CODE_ROOT)]
for module in list(sys.modules):
 if module in ('phase4_cjpe','phase3_rr') or module.startswith(('phase4_cjpe.','phase3_rr.')):del sys.modules[module]
importlib.invalidate_caches()
from phase4_cjpe.common import sha,atomic_json,source_hashes,now
print('Checking the existing 1.49 GB B1 input SHA; this may take a few minutes on Drive.',flush=True)
assert INPUT.stat().st_size==1491952478 and sha(INPUT)=='22b49e8f370e8f9bfc1a3f3f88cab813c0e7443edb9091e30f8d4877508690c9','B1 input upload is incomplete or different.'
DATA_ROOT=WORK/'Phase_4_CJPE'
from phase4_cjpe.data import acquire
missing=any(not (DATA_ROOT/p).is_file() for p in ('Train/train.json','Train/dev.json','Test/test.json'))
data_zip=next((p for p in (PERSIST/'inputs/Phase_4_CJPE.zip',Path('/content/Phase_4_CJPE.zip')) if p.is_file()),None)
try:DATA_ROOT=acquire(DATA_ROOT,uploaded_zip=data_zip if missing else None,download=bool(missing and not data_zip))
except Exception as exc:
 if getattr(getattr(exc,'response',None),'status_code',None) not in (401,403):raise
 from getpass import getpass
 from huggingface_hub import login
 print('Complete official-dataset authentication before leaving this running.',flush=True)
 token=getpass('Approved Hugging Face READ token (hidden): ')
 try:login(token=token,add_to_git_credential=False)
 finally:del token
 DATA_ROOT=acquire(DATA_ROOT,download=True)
CONFIG={'name':'02_B1_Partial_QueryHier','family':'b1','method':'partial','experiment':'adaptation',
 'rank':8,'alpha':16,'seed':781,'max_tokens':384,'chunks_per_case':8,'region_chars':8192,
 'max_train_cases':12000,'holdout_per_class':256,'max_epochs':4,'accumulation':8,
 'backbone_lr':2e-5,'head_lr':3e-4,'training_hours':4.,'eval_every':100,'save_every':25,'patience':6,
 'teacher_hours':1.5,'kd_temperature':2.,'kd_weight':.5,'stall_minutes':45,
 'export_reserve_minutes':75,'deadline_utc':'2026-10-21T18:29:00Z','allow_after_deadline':False,
 'run_root':str(RUN_ROOT),'work_root':str(WORK),'data_root':str(DATA_ROOT),'bundle':str(INPUT),
 'chunk_batch_size':2,'document_architecture':'b1_query_hierarchy_v1','label_smoothing':.02}
# On an exact resume keep the previously selected memory setting unchanged.
saved=RUN_ROOT/'worker_config.json'
if saved.exists():
 prior=json.loads(saved.read_text())
 assert {k:v for k,v in prior.items() if k!='chunk_batch_size'}=={k:v for k,v in CONFIG.items() if k!='chunk_batch_size'},'Run configuration changed. Preserve this run and choose a new run name.'
 CONFIG['chunk_batch_size']=prior['chunk_batch_size']
from phase4_cjpe.train import deadline_timestamp
if not (RUN_ROOT/'complete.json').exists():
 assert time.time()<deadline_timestamp(CONFIG)-CONFIG['export_reserve_minutes']*60,'Official training cutoff passed. No on-time submission will be fabricated.'
revision=DATA_ROOT/'download_revision.json'
if revision.exists():shutil.copy2(revision,RUN_ROOT/'official_download_revision.json')
atomic_json(RUN_ROOT/'design_and_retention.json',{'architecture':CONFIG['document_architecture'],
 'query_count':4,'encoder_layers_updated':2,'decoder_layers_updated':2,'windows':8,'tokens_per_window':384,
 'retention':'Only rolling latest.pt full resume and selected best.pt; atomic replacements, no numbered checkpoint history.',
 'original_input_and_other_runs_preserved':True,'source':source_hashes(),'recorded_utc':now()})

from phase3_rr.archive_inputs import resolve_bundle
from phase3_rr.data import Tokenizer
from phase4_cjpe.models import make_model
bundle,manifest=resolve_bundle(INPUT,WORK/'weights');assert manifest['family']=='b1'
tok=Tokenizer(bundle/'tokenizer.json')
sample='The Supreme Court considers the appeal under section १२३.'
ids=tok.encode(sample)
assert ids and tok.impl.decode(ids)==sample,'Original Unicode13 tokenizer round-trip failed.'

def gpu_preflight(batch_size):
 model=logits=loss=scaler=None
 try:
  model,evidence=make_model(bundle,'b1','partial',8,16);model.chunk_batch_size=batch_size
  raw=(ids*((382+len(ids)-1)//len(ids)))[:382]
  chunks=[[1]+raw+[2] for _ in range(8)];chunks[-1]=chunks[-1][:-64]+[2]
  tensor_bytes=sum(t.numel()*t.element_size() for t in model.state_dict().values())
  trainable_bytes=sum(p.numel()*p.element_size() for p in model.parameters() if p.requires_grad)
  peak=int(3*tensor_bytes+4*trainable_bytes+250e6)
  print('Estimated additional peak Drive space for atomic saves:',round(peak/1e9,2),'GB. Account quota is not independently verified.',flush=True)
  model=model.to('cuda').train()
  with torch.autocast(device_type='cuda',dtype=torch.float16):logits=model(chunks,[i/7 for i in range(8)],torch.device('cuda')).float()
  loss=torch.nn.functional.cross_entropy(logits,torch.tensor([0],device='cuda'))
  assert bool(torch.isfinite(loss)),'Nonfinite GPU preflight loss.'
  scaler=torch.amp.GradScaler('cuda',init_scale=128);scaler.scale(loss).backward()
  for group,prefixes in [('encoder',('backbone.encoder.',)),('decoder',('backbone.decoder.',)),('new head',('queries','token_pool.','chunk_projection.','document_encoder.','document_pool.','head.'))]:
   found=False
   for name,param in model.named_parameters():
    if name.startswith(prefixes) and param.requires_grad and param.grad is not None:
     assert bool(torch.isfinite(param.grad).all()),'Nonfinite '+group+' gradient: '+name
     if bool(param.grad.abs().max()>0):found=True
   assert found,'No nonzero '+group+' gradients.'
  result={'actual_gpu':torch.cuda.get_device_name(0),'finite_loss':float(loss.detach()),'parameters':evidence['parameters'],
   'trainable_parameters':evidence['trainable_parameters'],'chunk_batch_size':batch_size,'windows':8,'tokens_per_window':384,
   'encoder_decoder_and_head_gradients_verified':True,'estimated_additional_drive_peak_bytes':peak,
   'drive_quota_verified':False,'optimizer_steps':0,'checked_utc':now()}
  return result
 finally:
  del model,logits,loss,scaler;gc.collect();torch.cuda.empty_cache()

if not (RUN_ROOT/'complete.json').exists():
 print('Checking all eight 384-token windows on the real GPU before training.',flush=True)
 memory_fallback=False
 try:preflight=gpu_preflight(CONFIG['chunk_batch_size'])
 except torch.OutOfMemoryError:
  if CONFIG['chunk_batch_size']==1 or saved.exists():raise
  memory_fallback=True
 if memory_fallback:
  gc.collect();torch.cuda.empty_cache()
  CONFIG['chunk_batch_size']=1
  print('Preflight memory fallback: one chunk at a time, preserving all eight windows and context length.',flush=True)
  preflight=gpu_preflight(1)
 atomic_json(RUN_ROOT/'preflight.json',preflight)
 print('TOKENIZER + FULL-CONTEXT GPU PREFLIGHT PASSED:',preflight,flush=True)
print('Checkpoint policy: replace latest.pt and best.pt atomically; no epoch/step checkpoint history.',flush=True)
print('Preparing local SQLite caches, then training. Confirm increasing optimizer steps before leaving.',flush=True)
from phase4_cjpe.runtime import supervise
from phase4_cjpe.export import validate_zip
try:RESULT=supervise(CONFIG)
except Exception:
 failure=RUN_ROOT/'failure.json'
 if failure.exists():print('CURRENT FAILURE:\n'+json.loads(failure.read_text()).get('traceback',''))
 raise
expected_ids=json.loads((RUN_ROOT/'splits.json').read_text())['test']
checks=validate_zip(RESULT['submission'],expected_ids);assert checks['sha256']==RESULT['sha256']
READY=PERSIST/'SUBMISSIONS';READY.mkdir(parents=True,exist_ok=True)
SUBMISSION=READY/Path(RESULT['submission']).name
shutil.copy2(RESULT['submission'],SUBMISSION);assert sha(SUBMISSION)==RESULT['sha256']
atomic_json(READY/(SUBMISSION.stem+'_RESULT.json'),RESULT)
print('VERIFIED SUBMISSION ZIP:',SUBMISSION,flush=True)
print('LOCAL DEV MACRO F1:',RESULT['dev_metrics']['macro_f1'],flush=True)
print('NEW OPTIMIZER UPDATES:',RESULT['optimizer_updates'],flush=True)
print('COMPLETED AFTER DEADLINE:',RESULT['completed_after_deadline'],flush=True)
from IPython.display import FileLink,display
display(FileLink(str(SUBMISSION)))
# No duplicate model-weights archive: selected weights remain in best.pt.
records=RUN_ROOT/'EXACT_SOURCE_B1_QUERY_HIERARCHY.zip';records.write_bytes(source_bytes)
assert sha(records)==hashlib.sha256(source_bytes).hexdigest()
print('BEST MODEL:',RUN_ROOT/'best.pt',flush=True)
print('FULL OPTIMIZER RESUME:',RUN_ROOT/'latest.pt',flush=True)
print('Source, tokenizer, data audit, exact splits, selection, logs and probabilities are preserved in:',RUN_ROOT,flush=True)
